# Grid View

> Grid layout component for displaying media files as cards.

In [ ]:
#| default_exp components.grid_view

In [ ]:
#| export
from typing import Any, List, Optional

from fasthtml.common import Div, Span, Button, Img

# Tailwind utilities
from cjm_fasthtml_tailwind.utilities.sizing import w, h, min_h
from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.typography import (
    font_size, font_weight, truncate, text_align
)
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import (
    flex_display, flex_direction, items, justify, grid_display, grid_cols, gap
)
from cjm_fasthtml_tailwind.utilities.borders import border, rounded
from cjm_fasthtml_tailwind.utilities.interactivity import cursor
from cjm_fasthtml_tailwind.utilities.layout import position, top, right, left, z
from cjm_fasthtml_tailwind.core.base import combine_classes

# DaisyUI utilities
from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui, text_dui, border_dui
from cjm_fasthtml_daisyui.components.data_display.badge import badge

# Local imports
from cjm_file_discovery.core.models import FileInfo, FileType
from cjm_fasthtml_media_gallery.core.config import GalleryConfig, CardSize
from cjm_fasthtml_media_gallery.core.html_ids import GalleryHtmlIds
from cjm_fasthtml_media_gallery.core.icons import (
    get_media_type_icon, MEDIA_TYPE_BADGE_COLORS, get_gallery_icon
)

## Card Size Configuration

Tailwind classes for different card sizes.

In [ ]:
#| export
CARD_SIZE_CLASSES: dict[CardSize, dict[str, str]] = {
    CardSize.SMALL: {
        "card_height": str(h(32)),
        "icon_size": 6,
        "thumbnail_height": str(h(20)),
    },
    CardSize.MEDIUM: {
        "card_height": str(h(48)),
        "icon_size": 10,
        "thumbnail_height": str(h(32)),
    },
    CardSize.LARGE: {
        "card_height": str(h(64)),
        "icon_size": 14,
        "thumbnail_height": str(h(48)),
    },
}

## Media Card

Individual card component for a media file.

In [ ]:
#| export
def render_media_card(
    file_info: FileInfo,              # File to render
    config: GalleryConfig,            # Gallery configuration
    index: int,                       # Index in the grid
    file_url: Optional[str] = None,   # URL for thumbnail/preview
    is_selected: bool = False,        # Whether file is selected
    preview_url: Optional[str] = None,  # URL for preview action
    select_url: Optional[str] = None,   # URL for selection action
    hx_target: Optional[str] = None,    # HTMX target for swaps
) -> Any:  # Card component
    """Render a media file as a grid card."""
    size_config = CARD_SIZE_CLASSES[config.grid.card_size]
    can_select = config.can_select(file_info)
    
    # Card styling
    card_cls = combine_classes(
        "group relative",
        flex_display, flex_direction.col,
        bg_dui.base_100, rounded.lg,
        border(),
        border_dui.primary if is_selected else border_dui.base_300,
        "hover:border-primary hover:shadow-md",
        "transition-all duration-200",
        cursor.pointer if (config.preview.enable_preview or can_select) else None,
        "overflow-hidden"
    )
    
    item_id = GalleryHtmlIds.grid_item_id(index)
    
    # Card click action
    card_attrs = {
        "id": item_id,
        "cls": card_cls,
        "data-path": file_info.path,
        "data-index": str(index),
    }
    
    # Primary action: preview or select
    if config.preview.enable_preview and preview_url:
        card_attrs["hx_post"] = preview_url
        card_attrs["hx_vals"] = f'{{"path": "{file_info.path}"}}'
        card_attrs["hx_target"] = f"#{config.preview_modal_id}"
        card_attrs["hx_swap"] = "innerHTML"
    
    # Thumbnail or icon area
    thumbnail_area = _render_thumbnail_area(
        file_info, config, file_url, size_config
    )
    
    # File info area
    info_area = _render_info_area(file_info, config)
    
    # Selection indicator
    selection_indicator = None
    if can_select:
        selection_indicator = _render_selection_indicator(
            is_selected, select_url, file_info.path, hx_target
        )
    
    # Type badge
    type_badge = None
    if config.grid.show_file_type:
        type_badge = _render_type_badge(file_info)
    
    return Div(
        selection_indicator,
        type_badge,
        thumbnail_area,
        info_area,
        **card_attrs
    )

In [ ]:
#| export
def _render_thumbnail_area(
    file_info: FileInfo,              # File info
    config: GalleryConfig,            # Gallery config
    file_url: Optional[str],          # URL for image thumbnail
    size_config: dict,                # Size configuration
) -> Any:  # Thumbnail area component
    """Render the thumbnail/icon area of a card."""
    area_cls = combine_classes(
        flex_display, items.center, justify.center,
        w.full, size_config["thumbnail_height"],
        bg_dui.base_200,
        "overflow-hidden"
    )
    
    # Show thumbnail for images if enabled and URL available
    if (config.grid.show_thumbnails and 
        file_info.file_type == FileType.IMAGE and 
        file_url):
        return Div(
            Img(
                src=file_url,
                alt=file_info.name,
                cls=combine_classes(w.full, h.full, "object-cover"),
                loading="lazy"
            ),
            cls=area_cls
        )
    
    # Otherwise show file type icon
    return Div(
        get_media_type_icon(file_info.file_type, size=size_config["icon_size"]),
        cls=area_cls
    )

In [ ]:
#| export
def _render_info_area(
    file_info: FileInfo,              # File info
    config: GalleryConfig,            # Gallery config
) -> Any:  # Info area component
    """Render the file info area of a card."""
    info_cls = combine_classes(
        flex_display, flex_direction.col,
        p(3), gap(1)
    )
    
    # File name
    name = Span(
        file_info.name,
        cls=combine_classes(
            font_size.sm, font_weight.medium,
            truncate, text_dui.base_content
        ),
        title=file_info.name
    )
    
    # File size
    size = None
    if config.grid.show_file_size and file_info.size_str:
        size = Span(
            file_info.size_str,
            cls=combine_classes(
                font_size.xs, text_dui.base_content.opacity(60)
            )
        )
    
    return Div(name, size, cls=info_cls)

In [ ]:
#| export
def _render_selection_indicator(
    is_selected: bool,                # Current selection state
    select_url: Optional[str],        # URL for selection action
    file_path: str,                   # File path for HTMX
    hx_target: Optional[str],         # HTMX target
) -> Any:  # Selection indicator component
    """Render the selection indicator checkbox."""
    indicator_cls = combine_classes(
        position.absolute, top(2), left(2), z(10),
        w(6), h(6), rounded.full,
        flex_display, items.center, justify.center,
        bg_dui.primary if is_selected else bg_dui.base_100,
        border(), border_dui.base_300 if not is_selected else border_dui.primary,
        "opacity-0 group-hover:opacity-100",
        "opacity-100" if is_selected else None,  # Always show if selected
        "transition-opacity duration-200"
    )
    
    attrs = {
        "cls": indicator_cls,
        "onclick": "event.stopPropagation()",  # Don't trigger card click
    }
    
    if select_url:
        attrs["hx_post"] = select_url
        attrs["hx_vals"] = f'{{"path": "{file_path}"}}'
        if hx_target:
            attrs["hx_target"] = hx_target
        attrs["hx_swap"] = "outerHTML"
    
    return Div(
        get_gallery_icon("check", size=4, cls=str(text_dui.primary_content)) if is_selected else None,
        **attrs
    )

In [ ]:
#| export
def _render_type_badge(
    file_info: FileInfo,              # File info
) -> Any:  # Type badge component
    """Render the file type badge."""
    badge_color = MEDIA_TYPE_BADGE_COLORS.get(
        file_info.file_type, str(badge)
    )
    
    type_label = file_info.file_type.value.upper()
    if file_info.extension:
        type_label = file_info.extension.upper()
    
    return Span(
        type_label,
        cls=combine_classes(
            position.absolute, top(2), right(2),
            badge, badge_color,
            font_size.xs
        )
    )

In [ ]:
from fasthtml.common import to_xml

# Test media card
config = GalleryConfig()
file_info = FileInfo(
    name="photo.jpg", path="/photos/photo.jpg", is_directory=False,
    file_type=FileType.IMAGE, extension="jpg", size=1024000
)

card = render_media_card(
    file_info=file_info,
    config=config,
    index=0,
    preview_url="/preview"
)
html = to_xml(card)
assert "photo.jpg" in html
assert "media-gallery-item-0" in html
assert "hx-post" in html
assert "JPG" in html  # Type badge

# Test with selection enabled
select_config = GalleryConfig(
    selection_mode="single"
)
from cjm_fasthtml_media_gallery.core.config import SelectionMode
select_config = GalleryConfig(selection_mode=SelectionMode.SINGLE)
card = render_media_card(
    file_info=file_info,
    config=select_config,
    index=1,
    is_selected=True,
    select_url="/select"
)
html = to_xml(card)
assert "border-primary" in html  # Selected border

print("render_media_card tests passed!")

render_media_card tests passed!


## Grid View

Complete grid view with multiple cards.

In [ ]:
#| export
def render_grid_view(
    files: List[FileInfo],            # Files to display
    config: GalleryConfig,            # Gallery configuration
    get_file_url: Optional[callable] = None,  # Function to get file URL
    selected_paths: Optional[List[str]] = None,  # Currently selected paths
    preview_url: Optional[str] = None,  # URL for preview action
    select_url: Optional[str] = None,   # URL for selection action
    hx_target: Optional[str] = None,    # HTMX target for swaps
    start_index: int = 0,               # Starting index for IDs (for pagination)
) -> Any:  # Grid view component
    """Render a grid view of media files."""
    selected_paths = selected_paths or []
    
    # Grid layout
    columns = config.grid.columns
    grid_cls = combine_classes(
        grid_display, grid_cols(columns), gap(4),
        p(4)
    )
    
    # Render cards
    cards = []
    for i, file_info in enumerate(files):
        file_url = get_file_url(file_info.path) if get_file_url else None
        is_selected = file_info.path in selected_paths
        
        card = render_media_card(
            file_info=file_info,
            config=config,
            index=start_index + i,
            file_url=file_url,
            is_selected=is_selected,
            preview_url=preview_url,
            select_url=select_url,
            hx_target=hx_target
        )
        cards.append(card)
    
    return Div(
        *cards,
        id=GalleryHtmlIds.GRID,
        cls=grid_cls
    )

In [ ]:
# Test grid view
config = GalleryConfig()
files = [
    FileInfo(name="photo1.jpg", path="/photo1.jpg", is_directory=False, file_type=FileType.IMAGE, extension="jpg"),
    FileInfo(name="song.mp3", path="/song.mp3", is_directory=False, file_type=FileType.AUDIO, extension="mp3"),
    FileInfo(name="video.mp4", path="/video.mp4", is_directory=False, file_type=FileType.VIDEO, extension="mp4"),
]

grid = render_grid_view(
    files=files,
    config=config,
    preview_url="/preview"
)
html = to_xml(grid)
assert 'id="media-gallery-grid"' in html
assert "grid-cols-4" in html
assert "photo1.jpg" in html
assert "song.mp3" in html
assert "video.mp4" in html
assert "media-gallery-item-0" in html
assert "media-gallery-item-1" in html
assert "media-gallery-item-2" in html

# Test with different column count
from cjm_fasthtml_media_gallery.core.config import GridConfig
config_6col = GalleryConfig(grid=GridConfig(columns=6))
grid = render_grid_view(files=files, config=config_6col)
html = to_xml(grid)
assert "grid-cols-6" in html

# Test with selection
select_config = GalleryConfig(selection_mode=SelectionMode.MULTIPLE)
grid = render_grid_view(
    files=files,
    config=select_config,
    selected_paths=["/photo1.jpg"],
    select_url="/select"
)
html = to_xml(grid)
assert "border-primary" in html  # Selected item

print("render_grid_view tests passed!")

render_grid_view tests passed!


## Empty State

Component shown when there are no files to display.

In [ ]:
#| export
def render_grid_empty_state(
    message: str = "No media files found",  # Message to display
) -> Any:  # Empty state component
    """Render empty state for grid view."""
    return Div(
        Div(
            get_media_type_icon(FileType.IMAGE, size=16, with_color=False),
            cls=combine_classes(text_dui.base_content.opacity(30), m.b(4))
        ),
        Span(
            message,
            cls=combine_classes(font_size.lg, text_dui.base_content.opacity(60))
        ),
        id=GalleryHtmlIds.GRID,
        cls=combine_classes(
            flex_display, flex_direction.col, items.center, justify.center,
            min_h(64), p(8)
        )
    )

In [ ]:
# Test empty state
empty = render_grid_empty_state()
html = to_xml(empty)
assert "No media files found" in html
assert 'id="media-gallery-grid"' in html

# Test with custom message
empty = render_grid_empty_state("No images in this folder")
html = to_xml(empty)
assert "No images in this folder" in html

print("render_grid_empty_state tests passed!")

render_grid_empty_state tests passed!


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()